In [3]:
from pathlib import Path
import pandas as pd

# Load source tables
raw_path = Path("../raw").resolve()

customers = pd.read_csv(raw_path / "olist_customers_dataset.csv")
orders = pd.read_csv(raw_path / "olist_orders_dataset.csv")
order_items = pd.read_csv(raw_path / "olist_order_items_dataset.csv")
payments = pd.read_csv(raw_path / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(raw_path / "olist_order_reviews_dataset.csv")

# Convert order dates
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

# Keep delivered orders only
delivered_orders = orders.loc[orders["order_status"] == "delivered"].copy()

# Create order-level aggregates
items_by_order = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "count"),
        product_revenue=("price", "sum"),
        freight_value=("freight_value", "sum"),
    )
)

payments_by_order = (
    payments
    .groupby("order_id", as_index=False)
    .agg(payment_value=("payment_value", "sum"))
)

reviews_by_order = (
    reviews
    .groupby("order_id", as_index=False)
    .agg(review_score=("review_score", "mean"))
)

# Create one clean row per completed order
order_fact = (
    delivered_orders
    .merge(customers, on="customer_id", how="left", validate="many_to_one")
    .merge(items_by_order, on="order_id", how="left", validate="one_to_one")
    .merge(payments_by_order, on="order_id", how="left", validate="one_to_one")
    .merge(reviews_by_order, on="order_id", how="left", validate="one_to_one")
)

order_fact["order_value"] = (
    order_fact["product_revenue"] + order_fact["freight_value"]
)

order_fact["delivery_days"] = (
    order_fact["order_delivered_customer_date"]
    - order_fact["order_purchase_timestamp"]
).dt.total_seconds() / 86400

order_fact["is_late"] = (
    order_fact["order_delivered_customer_date"]
    > order_fact["order_estimated_delivery_date"]
)

# Create one row per real customer
customer_360 = (
    order_fact
    .sort_values("order_purchase_timestamp")
    .groupby("customer_unique_id", as_index=False)
    .agg(
        customer_state=("customer_state", "last"),
        first_purchase_date=("order_purchase_timestamp", "min"),
        last_purchase_date=("order_purchase_timestamp", "max"),
        total_orders=("order_id", "nunique"),
        total_revenue=("order_value", "sum"),
        average_order_value=("order_value", "mean"),
        average_review_score=("review_score", "mean"),
        average_delivery_days=("delivery_days", "mean"),
        late_order_rate=("is_late", "mean"),
    )
)

# Calculate RFM scores
reference_date = order_fact["order_purchase_timestamp"].max()

customer_360["recency_days"] = (
    reference_date - customer_360["last_purchase_date"]
).dt.days

customer_360["recency_score"] = pd.qcut(
    customer_360["recency_days"].rank(method="first"),
    q=4,
    labels=[4, 3, 2, 1],
).astype(int)

customer_360["frequency_score"] = pd.qcut(
    customer_360["total_orders"].rank(method="first"),
    q=4,
    labels=[1, 2, 3, 4],
).astype(int)

customer_360["monetary_score"] = pd.qcut(
    customer_360["total_revenue"].rank(method="first"),
    q=4,
    labels=[1, 2, 3, 4],
).astype(int)

# Assign customer segments
customer_360["customer_segment"] = "Inactive"

customer_360.loc[
    customer_360["recency_score"] >= 3,
    "customer_segment"
] = "Active"

customer_360.loc[
    (customer_360["recency_score"] == 4)
    & (customer_360["total_orders"] == 1),
    "customer_segment"
] = "New"

customer_360.loc[
    (customer_360["frequency_score"] >= 3)
    & (customer_360["monetary_score"] >= 3),
    "customer_segment"
] = "High Value"

customer_360.loc[
    (customer_360["recency_score"] == 1)
    & (customer_360["frequency_score"] >= 2),
    "customer_segment"
] = "At Risk"

# Assign a transparent customer-risk level
customer_360["risk_level"] = "Low"

customer_360.loc[
    (customer_360["recency_score"] <= 2)
    | (customer_360["average_review_score"] <= 2),
    "risk_level"
] = "Medium"

customer_360.loc[
    (customer_360["customer_segment"] == "At Risk")
    | (customer_360["average_review_score"] <= 1),
    "risk_level"
] = "High"

# Segment summary
segment_summary = (
    customer_360
    .groupby("customer_segment")
    .agg(
        customers=("customer_unique_id", "count"),
        revenue=("total_revenue", "sum"),
        average_order_value=("average_order_value", "mean"),
    )
    .round(2)
    .sort_values("revenue", ascending=False)
)

segment_summary

import duckdb

# Save Power BI-ready CSV files
processed_path = Path("../processed").resolve()
processed_path.mkdir(exist_ok=True)

order_fact.to_csv(
    processed_path / "order_fact.csv",
    index=False
)

customer_360.to_csv(
    processed_path / "customer_360.csv",
    index=False
)

# Create a local SQL database
database_path = Path("../olist_sales_intelligence.duckdb").resolve()

con = duckdb.connect(str(database_path))

con.register("order_fact_df", order_fact)
con.register("customer_360_df", customer_360)

con.execute("""
    CREATE OR REPLACE TABLE order_fact AS
    SELECT * FROM order_fact_df
""")

con.execute("""
    CREATE OR REPLACE TABLE customer_360 AS
    SELECT * FROM customer_360_df
""")

# Confirm the tables exist
con.execute("""
    SELECT
        customer_segment,
        COUNT(*) AS customers,
        ROUND(SUM(total_revenue), 2) AS revenue
    FROM customer_360
    GROUP BY customer_segment
    ORDER BY revenue DESC
""").df()



,customer_segment,customers,revenue
0,High Value,18151,4892380.15
1,Inactive,23325,3194370.14
2,At Risk,17440,2875774.65
3,New,17154,2250332.36
4,Active,17288,2206916.45


In [4]:
def create_recommended_action(row):
    if row["risk_level"] == "High" and row["average_review_score"] <= 2:
        return "Open a service-recovery case and contact the customer within 48 hours."

    if row["risk_level"] == "High":
        return "Prioritize a reactivation offer based on the customer's prior purchase value."

    if row["customer_segment"] == "High Value":
        return "Offer loyalty benefits or a relevant cross-sell recommendation."

    if row["customer_segment"] == "New":
        return "Send a post-purchase onboarding message and request feedback."

    if row["late_order_rate"] >= 0.10:
        return "Review delivery performance before the next customer contact."

    return "Maintain engagement with a relevant product recommendation."


def create_ai_summary(row):
    review_text = (
        f"{row['average_review_score']:.1f}/5"
        if pd.notna(row["average_review_score"])
        else "no review available"
    )

    return (
        f"{row['customer_segment']} customer in {row['customer_state']} with "
        f"{row['total_orders']} completed order(s) worth R${row['total_revenue']:.2f}. "
        f"Last purchase was {row['recency_days']} days before the dataset reference date. "
        f"Average review score: {review_text}. "
        f"Late-delivery rate: {row['late_order_rate'] * 100:.1f}%."
    )


customer_360["ai_customer_summary"] = customer_360.apply(
    create_ai_summary,
    axis=1
)

customer_360["recommended_next_action"] = customer_360.apply(
    create_recommended_action,
    axis=1
)

# Update the Power BI file and SQL table
processed_path = Path("../processed").resolve()
customer_360.to_csv(
    processed_path / "customer_360.csv",
    index=False
)

database_path = Path("../olist_sales_intelligence.duckdb").resolve()
con = duckdb.connect(str(database_path))

con.register("customer_360_df", customer_360)

con.execute("""
    CREATE OR REPLACE TABLE customer_360 AS
    SELECT * FROM customer_360_df
""")

con.close()

customer_360[
    [
        "customer_segment",
        "risk_level",
        "ai_customer_summary",
        "recommended_next_action",
    ]
].sample(5, random_state=42)

,customer_segment,risk_level,ai_customer_summary,recommended_next_action
21568,Active,Low,Active customer in MS with 1 completed order(s...,Maintain engagement with a relevant product re...
52028,New,Low,New customer in SP with 1 completed order(s) w...,Send a post-purchase onboarding message and re...
74245,High Value,Medium,High Value customer in SP with 2 completed ord...,Offer loyalty benefits or a relevant cross-sel...
45830,At Risk,High,At Risk customer in DF with 1 completed order(...,Prioritize a reactivation offer based on the c...
33285,New,Low,New customer in SP with 1 completed order(s) w...,Send a post-purchase onboarding message and re...
